# Pertemuan 4 — Data Cleaning dan Exploratory Data Analysis

Dataset yang sama dengan pertemuan 3 (`data.csv`), tapi sekarang kita bersihin.

## NumPy vs Pandas: Recap Singkat

Sesi membahas Pandas bersihin data (`drop_duplicates`, `fillna`, `to_datetime`) dan meringkasnya (`groupby`, `agg`, `pivot_table`). Untuk data visualisasi hanya akan dibahas di pertemuan 5 saja.

## 0. Sebelum ke Dataset Asli: Kenalan Dulu Sama Perintahnya

Data Cleaning itu bersihin data (buang duplikat, isi yang kosong, benerin tipe data, cek outlier) sebelum dipakai buat apa-apa.

EDA itu langkah sesudahnya — data yang udah bersih diringkas biar ketauan ceritanya (`groupby`, `agg`, `pivot_table`).

Urutannya selalu sama: beresin dulu datanya, baru gali insight-nya. Kalau kebalik, hasil ringkasannya masih kena efek data yang berantakan.

Biar kebayang gunanya masing-masing perintah, coba dulu di tabel kecil (`toy`, 6 baris, sengaja dibikin berantakan) — tiap perintah ditunjukin hasil **sebelum** dan **sesudah**, baru abis itu dipakai ke `data.csv` yang beneran.

In [ ]:
import pandas as pd

toy = pd.DataFrame({
    "id": [1, 2, 3, 4, 4, 5],
    "tanggal": ["2026-01-01", "2026-01-05", "31-02-2026", "2026-02-10", "2026-02-10", "2026-03-01"],
    "kategori": ["A", "B", "A", "B", "B", "A"],
    "harga": ["100000", "abc", "250000", "300000", "300000", "5000000"],
})
display(toy)

### Buang duplikat: `drop_duplicates()`

Baris id 4 muncul 2 kali persis sama. Kalau dibiarkan, nanti pas dijumlahkan jadi kehitung dobel.

In [ ]:
print("Sebelum, jumlah baris:", toy.shape[0])

toy = toy.drop_duplicates()

print("Sesudah, jumlah baris:", toy.shape[0])
display(toy)

### Benerin tipe data: `to_numeric()` / `to_datetime()`

Kolom `harga` sama `tanggal` sebenarnya harusnya angka dan tanggal, tapi kebaca sebagai teks biasa. Ada juga nilai yang formatnya rusak (`"abc"`, `"31-02-2026"`) — pakai `errors="coerce"` biar yang rusak itu jadi kosong aja, bukan bikin error.

In [ ]:
print("Sebelum:")
print(toy.dtypes)

toy["harga"] = pd.to_numeric(toy["harga"], errors="coerce")
toy["tanggal"] = pd.to_datetime(toy["tanggal"], errors="coerce")

print("\nSesudah:")
print(toy.dtypes)
display(toy)

### Kalau data kosong dihapus: `dropna()`

Ini contoh kalau kita PILIH buat menghapus baris yang ada kosongnya — bukan yang bakal dipakai, cuma dicoba dulu biar keliatan efeknya.

In [ ]:
print("Sebelum:")
display(toy)

print("Kalau dropna:")
display(toy.dropna())

### Isi data kosong: fillna()

Harga yang kosong diisi pakai median dari harga yang lain — masih masuk akal ditebak.

Tanggal beda kasus: gak ada tanggal yang bisa ditebak dengan masuk akal, jadi dibiarin NaT aja (versi "kosong" punya tanggal). Cukup dicatat berapa banyak yang kosong.


In [ ]:
print("Sebelum:")
display(toy)

print("Jumlah kosong per kolom:")
print(toy.isna().sum())

toy["harga"] = toy["harga"].fillna(toy["harga"].median())

print("Sesudah (harga diisi, tanggal dibiarin NaT):")
display(toy)

### Outlier: aturan IQR

Ada 2 harga yang kelewat jauh dari yang lain (100.000 dan 5.000.000). Coba lihat efeknya ke rata-rata sebelum dan sesudah dibuang.

In [ ]:
print("Sebelum, rata-rata harga:", toy["harga"].mean())

q1, q3 = toy["harga"].quantile(0.25), toy["harga"].quantile(0.75)
iqr = q3 - q1
batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
toy = toy[(toy["harga"] >= batas_bawah) & (toy["harga"] <= batas_atas)]

print("Sesudah, rata-rata harga:", toy["harga"].mean())
display(toy)

### Ringkas per kelompok: `groupby()` + `agg()` + `sort_values()`

Ini udah masuk EDA — datanya sudah bersih dari langkah-langkah di atas, sekarang diringkas per kategori.

In [ ]:
print("Sebelum diringkas:")
display(toy[["kategori", "harga"]])

ringkasan_toy = toy.groupby("kategori", as_index=False).agg(
    total=("harga", "sum"), jumlah=("id", "nunique")
).sort_values("total", ascending=False)

print("Sesudah diringkas:")
display(ringkasan_toy)

### Ringkasan 2 dimensi: `pivot_table()`

Data yang sama, cuma disusun ulang jadi tabel kategori × id.

In [ ]:
display(pd.pivot_table(toy, values="harga", index="kategori", columns="id", aggfunc="sum", fill_value=0))

### Filter

Ambil baris yang kategorinya "B" doang.

In [ ]:
print("Sebelum, jumlah baris:", toy.shape[0])

hasil_filter = toy[toy["kategori"] == "B"]

print("Sesudah difilter, jumlah baris:", hasil_filter.shape[0])
display(hasil_filter)

Segitu dulu contoh sederhananya. Sekarang perintah yang sama dipakai ke `data.csv` — dataset asli 133 baris dari pertemuan 3.

## 1. Recap Kondisi Data (Temuan Pertemuan 3)

In [ ]:
import pandas as pd

df = pd.read_csv("data.csv")
print(df.shape)
print(df.isna().sum())
print(df.duplicated().sum())

## 2. Menghapus Baris Duplikat

In [ ]:
baris_duplikat = df[df.duplicated(keep=False)].sort_values("id_pesanan")
display(baris_duplikat[["id_pesanan", "nama_produk", "kategori", "harga"]])

In [ ]:
df = df.drop_duplicates()
print(df.shape)
print(df.duplicated().sum())

## 3. Contoh Kecil `errors="coerce"`

In [ ]:
tanggal_kotor = pd.Series(["2026-01-15", "31-02-2026", "2026-03-10", "bukan tanggal"])
print(pd.to_datetime(tanggal_kotor, errors="coerce"))

harga_kotor = pd.Series(["150000", "abc", "220000", "75rb"])
print(pd.to_numeric(harga_kotor, errors="coerce"))

## 4. Konversi `tanggal` di Data Asli

In [ ]:
print(df["tanggal"].dtype)
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")
print(df["tanggal"].dtype)
print(df["tanggal"].isna().sum())

## 5. Melihat Baris `harga` Kosong

In [ ]:
kosong = df[df["harga"].isna()]
display(kosong[["id_pesanan", "nama_produk", "kategori", "harga"]])

## 6. Membandingkan Opsi Penanganan (Belum Dieksekusi)

In [ ]:
print("Kalau dihapus, shape jadi:", df.dropna(subset=["harga"]).shape)
print("Rata-rata (mean) harga:", df["harga"].mean())

for produk in kosong["nama_produk"].unique():
    median_produk = df[df["nama_produk"] == produk]["harga"].median()
    print(f"Median harga {produk}: {median_produk}")

## 7. Mengisi `harga` dengan Median per Produk

In [ ]:
df["harga"] = df["harga"].fillna(df.groupby("nama_produk")["harga"].transform("median"))
print(df["harga"].isna().sum())
display(df.loc[kosong.index, ["id_pesanan", "nama_produk", "harga"]])

## 8. Mendeteksi Outlier dengan Aturan IQR

In [ ]:
q1 = df["harga"].quantile(0.25)
q3 = df["harga"].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr)
print("Batas bawah:", batas_bawah, "| Batas atas:", batas_atas)

outlier = df[(df["harga"] < batas_bawah) | (df["harga"] > batas_atas)]
display(outlier[["id_pesanan", "nama_produk", "kategori", "harga", "modal"]])

## 9. Menyelidiki dan Memutuskan

In [ ]:
sneakers_lain = df[df["nama_produk"] == "Sepatu Sneakers"]
display(sneakers_lain[["id_pesanan", "harga", "modal"]])

In [ ]:
df = df[df["id_pesanan"] != 1041]
print(df.shape)

## 10. Verifikasi Kondisi Akhir

In [ ]:
print(df.shape)
print(df.isna().sum())
print(df.duplicated().sum())
print(df.dtypes)

## 11. Membuat Ulang Kolom Turunan

In [ ]:
df["nilai_penjualan"] = df["harga"] * df["jumlah"]
df["untung"] = (df["harga"] - df["modal"]) * df["jumlah"]
display(df[["nama_produk", "harga", "modal", "jumlah", "nilai_penjualan", "untung"]].head())

## 12. `groupby()` Pertama: Total Penjualan per Kategori

In [ ]:
display(df.groupby("kategori")["nilai_penjualan"].sum())

## 13. `agg()` dan `sort_values()`: Ringkasan Lengkap

In [ ]:
ringkasan = (
    df.groupby("kategori", as_index=False)
      .agg(total_penjualan=("nilai_penjualan", "sum"),
           jumlah_transaksi=("id_pesanan", "nunique"),
           rata_harga=("harga", "mean"))
      .sort_values("total_penjualan", ascending=False)
)
display(ringkasan)

## 14. EDA 1 — Kategori Penyumbang Penjualan Terbesar

In [ ]:
print("Berdasarkan total penjualan:")
display(ringkasan.sort_values("total_penjualan", ascending=False))

print("\nBerdasarkan jumlah transaksi:")
display(ringkasan.sort_values("jumlah_transaksi", ascending=False))

## 15. EDA 2 — Tren Penjualan per Bulan

In [ ]:
df["bulan"] = df["tanggal"].dt.to_period("M").astype(str)
per_bulan = df.groupby("bulan", as_index=False).agg(
    total_penjualan=("nilai_penjualan", "sum"),
    jumlah_transaksi=("id_pesanan", "nunique"),
)
display(per_bulan)

## 16. EDA 3 — Wilayah Ramai vs Wilayah Bernilai Tinggi

In [ ]:
per_wilayah = df.groupby("wilayah", as_index=False).agg(
    total_penjualan=("nilai_penjualan", "sum"),
    jumlah_transaksi=("id_pesanan", "nunique"),
)
per_wilayah["rata_rata_per_transaksi"] = (
    per_wilayah["total_penjualan"] / per_wilayah["jumlah_transaksi"]
).round(0)
display(per_wilayah.sort_values("jumlah_transaksi", ascending=False))

## 17. `pivot_table()`: Wilayah × Kategori

In [ ]:
pivot = pd.pivot_table(
    df, values="nilai_penjualan", index="wilayah", columns="kategori",
    aggfunc="sum", fill_value=0,
)
display(pivot)

## 18. Menguji Jebakan: Cek Kontribusi Satu Baris

In [ ]:
laptop = df[df["nama_produk"] == "Laptop Gaming (Bonus)"]
display(laptop[["id_pesanan", "nama_produk", "wilayah", "bulan", "nilai_penjualan"]])

bandung_total = df[df["wilayah"] == "Bandung"]["nilai_penjualan"].sum()
bandung_tanpa_laptop = df[
    (df["wilayah"] == "Bandung") & (df["nama_produk"] != "Laptop Gaming (Bonus)")
]["nilai_penjualan"].sum()
print("Bandung total:", bandung_total, "| tanpa laptop:", bandung_tanpa_laptop)

jan_total = df[df["bulan"] == "2026-01"]["nilai_penjualan"].sum()
jan_tanpa_laptop = df[
    (df["bulan"] == "2026-01") & (df["nama_produk"] != "Laptop Gaming (Bonus)")
]["nilai_penjualan"].sum()
print("Januari total:", jan_total, "| tanpa laptop:", jan_tanpa_laptop)

## 19. Menyimpan Data Bersih ke CSV Baru

In [ ]:
df_final = df.drop(columns=["bulan"])
df_final.to_csv("data-bersih.csv", index=False)
print("Tersimpan:", df_final.shape)

## 20. Latihan Gabungan

In [ ]:
print("Kolom numerik:")
print(df.select_dtypes(include="number").columns.tolist())

print("\nJumlah baris kategori Fashion:")
print(df[df["kategori"] == "Fashion"].shape)

print("\nRingkasan total penjualan per kategori, terurut dari terbesar:")
display(df.groupby("kategori", as_index=False)["nilai_penjualan"].sum()
          .sort_values("nilai_penjualan", ascending=False))

## Selesai

Data sudah dibersihkan (129 baris, 0 kosong, 0 duplikat) dan sudah dijawab 3 pertanyaan EDA lewat `groupby`/`agg`/`pivot_table`. Hasilnya tersimpan di `data-bersih.csv`, dipakai pertemuan 5 untuk visualisasi.